# Proyecto 3er Parcial — Análisis Quejas Habitacionales (Databricks)

**Materia:** Tópicos de Big Data · **Periodo:** agosto-diciembre de 2026 · **Tecnologías:** Databricks Free Edition, PySpark, Spark SQL, Delta Lake, Unity Catalog y Tableau Public.

**Equipo:** Issabela Marian Romero Aguayo, Deborah Villicaña Prieto


## Etapa 1 · Ingesta y preparación

Importación de librerías

In [0]:
import os
import math
import time
import requests
from pyspark.sql import functions as F


In [0]:
URL_BASE = "https://data.cityofnewyork.us/resource"
APP_TOKEN = None          # opcional: "tu_app_token"
HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}

INICIO = "2025-01-01T00:00:00"
FIN    = "2026-01-01T00:00:00"   # límite exclusivo: incluye todo el 31/12/2025

# Fuente 1: 311 Service Requests (solo quejas de vivienda atendidas por HPD)
ID_311 = "erm2-nwe9"
COLUMNAS_311 = [
    "unique_key", "created_date", "closed_date", "agency", "complaint_type",
    "descriptor", "location_type", "incident_zip", "borough", "status",
    "resolution_action_updated_date", "community_board", "bbl",
    "latitude", "longitude", "open_data_channel_type"
]
WHERE_311 = f"agency = 'HPD' AND created_date >= '{INICIO}' AND created_date < '{FIN}'"

# Fuente 2: HPD Housing Maintenance Code Violations
ID_VIOL = "wvxf-dwi5"
COLUMNAS_VIOL = [
    "violationid", "buildingid", "boroid", "boro", "zip", "class",
    "inspectiondate", "approveddate", "novissueddate", "novtype",
    "currentstatus", "currentstatusdate", "violationstatus", "rentimpairing",
    "communityboard", "councildistrict", "bbl", "nta", "latitude", "longitude"
]
WHERE_VIOL = f"novissueddate >= '{INICIO}' AND novissueddate < '{FIN}'"

# Catálogo de referencia: población por distrito
ID_POB = "xywu-7bv9"

Prueba la conexión con una consulta pequeña (`count(*)`). Un código HTTP 200 indica que Databricks puede llegar a la API. 

In [0]:
#def contar_registros(dataset_id, where):
#    """Cuenta en el servidor cuántas filas cumplen el filtro."""
#    r = requests.get(
#        f"{URL_BASE}/{dataset_id}.json",
#        params={"$select": "count(*) AS total", "$where": where},
#        headers=HEADERS,
#        timeout=60
#    )
#    r.raise_for_status()
#    return int(r.json()[0]["total"])


#print("Quejas 311 (HPD, 2025):", f"{contar_registros(ID_311, WHERE_311):,}")
#print("Violaciones HPD (2025):", f"{contar_registros(ID_VIOL, WHERE_VIOL):,}")

#NO FUNCIONÓOOO

Crea el catálogo, los esquemas `stage`, `clean` y `datamart`, y el volumen donde se guardarán los archivos de origen.

In [0]:
CATALOGO = "nyc_vivienda"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for esquema in ["stage", "clean", "datamart"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{esquema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.stage.archivos_origen")

RUTA_VOLUMEN = f"/Volumes/{CATALOGO}/stage/archivos_origen"
CARPETA_311  = f"{RUTA_VOLUMEN}/quejas_311_hpd_2025"
CARPETA_VIOL = f"{RUTA_VOLUMEN}/violaciones_hpd_2025"
CARPETA_POB  = f"{RUTA_VOLUMEN}/poblacion_distrito"

spark.sql(f"SHOW SCHEMAS IN {CATALOGO}").show(truncate=False)

In [0]:
# Genera las URL para descarga manual (solo si hace falta el plan B)
from urllib.parse import urlencode

def url_api(dataset_id, columnas, where, limite=1_000_000):
    params = {"$select": ",".join(columnas), "$where": where, "$limit": limite}
    return f"{URL_BASE}/{dataset_id}.csv?" + urlencode(params)

print(url_api(ID_311, COLUMNAS_311, WHERE_311), "\n")
print(url_api(ID_VIOL, COLUMNAS_VIOL, WHERE_VIOL), "\n")
print(url_api(ID_POB, ["age_group", "borough", "_2020"], "age_group = 'Total Population'"))

Crea las subcarpetas dentro del volumen con esta celda nueva

In [0]:
for carpeta in [CARPETA_311, CARPETA_VIOL, CARPETA_POB]:
    os.makedirs(carpeta, exist_ok=True)
    print("Carpeta lista:", carpeta)

In [0]:
for carpeta in [CARPETA_311, CARPETA_VIOL, CARPETA_POB]:
    for archivo in os.listdir(carpeta):
        tamano_mb = os.path.getsize(f"{carpeta}/{archivo}") / (1024 ** 2)
        print(f"{carpeta.split('/')[-1]}/{archivo}: {tamano_mb:.1f} MB")

## Etapa 2 · Lectura y exploración de los viajes

In [0]:
def leer_csv(carpeta):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(f"{carpeta}/*.csv")
    )

df_311_raw  = leer_csv(CARPETA_311)
df_viol_raw = leer_csv(CARPETA_VIOL)
df_pob_raw  = leer_csv(CARPETA_POB)

# Referencia: conteos que devolvía la API para 2025 (pueden variar un poco por actualizaciones)
print(f"Quejas 311 leídas:  {df_311_raw.count():,} (referencia ≈ 774,590)")
print(f"Violaciones leídas: {df_viol_raw.count():,} (referencia ≈ 868,018)")

In [0]:
import os

ruta_volumen = (
    "/Volumes/nyc_vivienda/stage/"
    "archivos_origen/"
)

archivos = [
    "quejas_311_hpd_2025/erm2-nwe9.csv",
    "violaciones_hpd_2025/wvxf-dwi5.csv",
    "poblacion_distrito/xywu-7bv9.csv"
]

for archivo in archivos:
    tamano_mb = os.path.getsize(ruta_volumen + archivo) / (1024 ** 2)
    print(f"{archivo}: {tamano_mb:.2f} MB")


tamano_kb = os.path.getsize(ruta_volumen + "poblacion_distrito/xywu-7bv9.csv") / 1024
print(f"Población: {tamano_kb:.2f} KB")

## Etapa 3 · Catálogo de zonas

## Etapa 4 · Diagnóstico de calidad

## Transformación con PySpark
Crear el DataFrame de viajes limpios

## Etapa 5 · Limpieza con PySpark

## Etapa 6 · Integración geográfica

## Etapa 7 · Persistencia con Delta Lake

## Etapa 8 · Datamart y análisis de demanda

## Etapa 9 · Preparación de datos para Tableau Public

## Etapa 10 · Construcción del dashboard en Tableau Public

**Exporta únicamente los resultados agregados**, no los 10.8 millones de viajes. Desde cada `display(...)` en Databricks utiliza la descarga CSV del resultado (si la interfaz no lo permite, exporta los DataFrames agregados a un volumen).

1. **Evolución diaria:** `demanda_diaria.csv`. Columnas: `fecha_viaje` como fecha continua; Filas: `SUM(total_viajes)`; Marcas: línea.
2. **Demanda por día:** misma fuente diaria. Columnas: `dia_semana` discreto; Filas: `AVG(total_viajes)`; Marcas: barras. Crea alias de domingo a sábado.
3. **Mapa de calor:** `demanda_horaria.csv`. Columnas: `hora` discreta; Filas: `dia_semana` discreto; Marcas: cuadrado; Color: promedio de viajes por día y hora. Para este trimestre, divide por 12 los martes y por 13 los demás días, o utiliza el número de ocurrencias obtenido desde `demanda_diaria`.
4. **Top 10 zonas:** `demanda_por_zona.csv`. Filas: `zona`; Columnas: `SUM(total_viajes)`; barras horizontales; filtro Top 10 por suma de viajes.

**Dashboard:** crea un dashboard nuevo en el **mismo libro**; coloca la evolución diaria arriba, barras semanales y ranking al centro, y mapa de calor abajo. Añade título, fuente, periodo y notas metodológicas. Las fuentes agregadas son diferentes: los filtros entre hojas no funcionan automáticamente sin relaciones/acciones configuradas.

### Entregables
- Notebook ejecutado con resultados y respuestas a las preguntas.
- Captura de las validaciones principales y explicación de las reglas de limpieza.
- Cuatro hojas y un dashboard publicado en Tableau Public.
- Conclusiones: tres hallazgos sustentados por datos y dos limitaciones del análisis.

### Reflexión final
¿Por qué la ingesta se realizó en Databricks y no en la laptop? ¿Qué diferencia hay entre una vista temporal y una tabla Delta? ¿Por qué no eliminamos automáticamente tarifas negativas ni valores superiores al P99? ¿Por qué usamos `AVG(total_viajes)` para comparar días de la semana?
